# SoundTag AST Hybrid v2

**26개 장르** — FSLD 드럼루프(wav) + Deezer 프리뷰(mp3) 하이브리드 데이터셋

- FSLD: 1,584 트랙 (Electropop, Dubstep, Breakbeat, Hip Hop, Techno, House, Pop Rock, DnB, Trance, Funk Pop, Trap, Lo-fi, Disco)
- Deezer: 1,600 트랙 (R&B, Latin, Ballad, City Pop, Afrobeats, Reggaeton, Moombahton, etc.)
- 총 3,184 트랙

**Add Input 필수:**
- `soundtag-hybrid-v2` (하이브리드 데이터)
- `soundtag-kpop-previews` (K-pop 분석용)

In [1]:
# ═══ 셀 1: 설치 + Import ═══
!pip install -q transformers accelerate torchaudio

import os, json, glob
import numpy as np
import torch
import torchaudio
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import ASTForAudioClassification, ASTFeatureExtractor

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else '')
print(f'PyTorch: {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')

GPU: Tesla T4

PyTorch: 2.10.0+cu128
torchaudio: 2.10.0+cu128


In [2]:
# ═══ 셀 2: 데이터 경로 설정 ═══
import os
from pathlib import Path

# Deezer 프리뷰 (기존 merged에서 deezer_ 파일만)
MERGED_DIR = '/kaggle/input/datasets/zoebae/soundtag-hybrid-v3/merged'

# Jamendo drums stem (s1 + s3)
DRUMS_DIRS = [
    '/kaggle/input/datasets/zoebae/soundtag-jamendo-drums-s1',
    '/kaggle/input/datasets/zoebae/soundtag-jamendo-drums-s3',
]

print(f'MERGED: {MERGED_DIR} → {os.path.exists(MERGED_DIR)}')
for d in DRUMS_DIRS:
    count = sum(1 for f in Path(d).rglob('*_drums.mp3')) if os.path.exists(d) else 0
    print(f'DRUMS: {d} → {count}개')

MERGED: /kaggle/input/datasets/zoebae/soundtag-hybrid-v3/merged → True
DRUMS: /kaggle/input/datasets/zoebae/soundtag-jamendo-drums-s1 → 2035개
DRUMS: /kaggle/input/datasets/zoebae/soundtag-jamendo-drums-s3 → 279개


In [3]:
# ═══ 셀 3: 파일 수집 ═══
def collect_audio_files(base_dirs):
    genre_files = {}
    for base in base_dirs:
        if not os.path.exists(base):
            continue
        for genre_dir in sorted(Path(base).iterdir()):
            if not genre_dir.is_dir():
                continue
            genre_name = genre_dir.name.replace('_', ' ').replace('and', '&').replace('RandB', 'R&B')
            files = list(genre_dir.glob('*.wav')) + list(genre_dir.glob('*.mp3'))
            files = [f for f in files if not f.name.startswith('._') 
                     and ('_drums' in f.name or f.name.startswith('fsld_'))]
            if files:
                if genre_name not in genre_files:
                    genre_files[genre_name] = []
                genre_files[genre_name].extend(files)
    return genre_files

# FSLD + Jamendo drums stem + Deezer 수집
genre_files = collect_audio_files([MERGED_DIR] + DRUMS_DIRS)

print(f'\n발견된 장르: {len(genre_files)}개')
total = 0
for genre, files in sorted(genre_files.items(), key=lambda x: -len(x[1])):
    print(f'  {genre:25s} {len(files):4d}개')
    total += len(files)
print(f'\n총: {total}개 파일')



발견된 장르: 20개
  Breakbeat                  497개
  Hip Hop                    468개
  House                      314개
  DnB                        280개
  Dubstep                    267개
  EDM                        265개
  Funk                       252개
  Techno                     227개
  Latin Pop                  209개
  Jazz                       200개
  Trance                     185개
  Electropop                 175개
  Pop Rock                   120개
  Disco                       90개
  Contemporary R&B            37개
  Trip hop                    32개
  Trap                        30개
  Lo fi                       20개
  Dancehall                    2개
  Neo Soul                     2개

총: 3672개 파일


In [4]:
# ═══ 셀 4: Dataset + Train/Val 분리 ═══

# 파일 리스트 + 레이블 생성
all_files = []
all_labels = []
for genre, files in genre_files.items():
    for f in files:
        all_files.append(str(f))
        all_labels.append(genre)

# LabelEncoder
le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
num_labels = len(le.classes_)

print(f'총 {len(all_files)}개 파일, {num_labels}개 장르')
print(f'장르: {list(le.classes_)}')

# 극소 장르 제외 (학습 불안정)
MIN_SAMPLES = 10
genre_files = {g: f for g, f in genre_files.items() if len(f) >= MIN_SAMPLES}

# Train/Val 분리 (80/20, stratified)
train_files, val_files, train_labels, val_labels = train_test_split(
    all_files, all_labels_enc, test_size=0.2,
    stratify=all_labels_enc, random_state=42
)

print(f'\nTrain: {len(train_files)}, Val: {len(val_files)}')

총 3672개 파일, 20개 장르
장르: [np.str_('Breakbeat'), np.str_('Contemporary R&B'), np.str_('Dancehall'), np.str_('Disco'), np.str_('DnB'), np.str_('Dubstep'), np.str_('EDM'), np.str_('Electropop'), np.str_('Funk'), np.str_('Hip Hop'), np.str_('House'), np.str_('Jazz'), np.str_('Latin Pop'), np.str_('Lo fi'), np.str_('Neo Soul'), np.str_('Pop Rock'), np.str_('Techno'), np.str_('Trance'), np.str_('Trap'), np.str_('Trip hop')]

Train: 2937, Val: 735


In [5]:
# ═══ 셀 5: PyTorch Dataset 클래스 ═══

class AudioDataset(Dataset):
    def __init__(self, files, labels, feature_extractor, sr=16000, duration=10):
        self.files = files
        self.labels = labels
        self.fe = feature_extractor
        self.sr = sr
        self.max_len = sr * duration
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]
        
        try:
            waveform, orig_sr = torchaudio.load(path)
        except Exception:
            # 읽기 실패 시 빈 오디오
            waveform = torch.zeros(1, self.max_len)
            orig_sr = self.sr
        
        # 모노 변환
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform.squeeze(0)
        
        # 리샘플링
        if orig_sr != self.sr:
            waveform = torchaudio.transforms.Resample(orig_sr, self.sr)(waveform)
        
        # 길이 맞추기
        if len(waveform) > self.max_len:
            waveform = waveform[:self.max_len]
        elif len(waveform) < self.max_len:
            waveform = torch.nn.functional.pad(waveform, (0, self.max_len - len(waveform)))
        
        # AST Feature Extractor
        inputs = self.fe(waveform.numpy(), sampling_rate=self.sr, return_tensors='pt')
        input_values = inputs['input_values'].squeeze(0)
        
        return {'input_values': input_values, 'labels': torch.tensor(label, dtype=torch.long)}

# Feature Extractor 로드
fe = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')

train_dataset = AudioDataset(train_files, train_labels, fe)
val_dataset = AudioDataset(val_files, val_labels, fe)

# 테스트: 첫 샘플 로드
sample = train_dataset[0]
print(f'input_values shape: {sample["input_values"].shape}')
print(f'label: {sample["labels"]} = {le.classes_[sample["labels"]]}')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

input_values shape: torch.Size([1024, 128])
label: 17 = Trance


In [6]:
# ═══ 셀 6: Weighted Sampler (클래스 불균형 해결) ═══

# 클래스별 가중치
label_counts = Counter(train_labels.tolist())
weights_per_class = {cls: 1.0 / count for cls, count in label_counts.items()}
sample_weights = [weights_per_class[label] for label in train_labels.tolist()]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print('클래스별 트랙 수 (train):')
for cls_idx, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    genre_name = le.classes_[cls_idx]
    weight = weights_per_class[cls_idx]
    print(f'  {genre_name:25s} {count:4d}개  (weight: {weight:.4f})')

클래스별 트랙 수 (train):
  Breakbeat                  397개  (weight: 0.0025)
  Hip Hop                    374개  (weight: 0.0027)
  House                      251개  (weight: 0.0040)
  DnB                        224개  (weight: 0.0045)
  Dubstep                    213개  (weight: 0.0047)
  EDM                        212개  (weight: 0.0047)
  Funk                       201개  (weight: 0.0050)
  Techno                     182개  (weight: 0.0055)
  Latin Pop                  167개  (weight: 0.0060)
  Jazz                       160개  (weight: 0.0063)
  Trance                     148개  (weight: 0.0068)
  Electropop                 140개  (weight: 0.0071)
  Pop Rock                    96개  (weight: 0.0104)
  Disco                       72개  (weight: 0.0139)
  Contemporary R&B            30개  (weight: 0.0333)
  Trip hop                    26개  (weight: 0.0385)
  Trap                        24개  (weight: 0.0417)
  Lo fi                       16개  (weight: 0.0625)
  Neo Soul                     2개  (weight: 0

In [7]:
# ═══ 셀 7: AST 모델 로드 + 학습 ═══

# 모델 로드 (classifier head를 26개로 교체)
model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)
model = model.cuda()

print(f'모델 로드 완료: {num_labels}개 장르 분류')

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# Optimizer + Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

NUM_EPOCHS = 10
best_acc = 0
results_log = []

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([20, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([20])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


모델 로드 완료: 20개 장르 분류


In [8]:
# ═══ 셀 8: 학습 루프 ═══

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_idx, batch in enumerate(train_loader):
        input_values = batch['input_values'].cuda()
        labels = batch['labels'].cuda()
        
        outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch+1} [{batch_idx+1}/{len(train_loader)}] loss: {loss.item():.4f}')
    
    scheduler.step()
    train_acc = train_correct / train_total
    avg_loss = train_loss / len(train_loader)
    
    # ── Validation ──
    model.eval()
    val_correct = 0
    val_total = 0
    all_preds = []
    all_true = []
    all_probs = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_values = batch['input_values'].cuda()
            labels = batch['labels'].cuda()
            
            outputs = model(input_values=input_values)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = probs.argmax(dim=-1)
            
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    val_acc = val_correct / val_total
    
    # Top-3 accuracy
    all_probs_np = np.array(all_probs)
    all_true_np = np.array(all_true)
    top3_preds = np.argsort(all_probs_np, axis=1)[:, -3:]
    top3_acc = np.mean([all_true_np[i] in top3_preds[i] for i in range(len(all_true_np))])
    
    log = f'Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {avg_loss:.4f}, Train: {train_acc:.1%}, Val: {val_acc:.1%}, Top-3: {top3_acc:.1%}'
    print(f'\n{log}\n')
    results_log.append(log)
    
    # Best 모델 저장
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/ast_hybrid_best.pt')
        print(f'  💾 Best model saved! ({val_acc:.1%})')

print(f'\n✅ 학습 완료! Best accuracy: {best_acc:.1%}')

  Epoch 1 [50/368] loss: 2.4356
  Epoch 1 [100/368] loss: 2.0576
  Epoch 1 [150/368] loss: 2.2851
  Epoch 1 [200/368] loss: 1.5335
  Epoch 1 [250/368] loss: 1.3494
  Epoch 1 [300/368] loss: 1.2357
  Epoch 1 [350/368] loss: 1.7170

Epoch 1/10 — Loss: 1.7627, Train: 49.1%, Val: 30.3%, Top-3: 58.8%

  💾 Best model saved! (30.3%)
  Epoch 2 [50/368] loss: 1.0209
  Epoch 2 [100/368] loss: 1.3266
  Epoch 2 [150/368] loss: 1.4080
  Epoch 2 [200/368] loss: 1.5500
  Epoch 2 [250/368] loss: 0.8370
  Epoch 2 [300/368] loss: 0.4773
  Epoch 2 [350/368] loss: 0.3200

Epoch 2/10 — Loss: 0.9654, Train: 73.2%, Val: 38.5%, Top-3: 64.4%

  💾 Best model saved! (38.5%)
  Epoch 3 [50/368] loss: 1.5415
  Epoch 3 [100/368] loss: 0.2595
  Epoch 3 [150/368] loss: 1.1696
  Epoch 3 [200/368] loss: 0.5034
  Epoch 3 [250/368] loss: 0.8573
  Epoch 3 [300/368] loss: 1.3875
  Epoch 3 [350/368] loss: 0.1477

Epoch 3/10 — Loss: 0.6737, Train: 82.4%, Val: 40.1%, Top-3: 64.6%

  💾 Best model saved! (40.1%)
  Epoch 4 [50/36

In [9]:
# ═══ 셀 9: 최종 평가 + Classification Report ═══

# Best 모델 로드
model.load_state_dict(torch.load('/kaggle/working/ast_hybrid_best.pt'))
model.eval()

all_preds = []
all_true = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        input_values = batch['input_values'].cuda()
        labels = batch['labels'].cuda()
        outputs = model(input_values=input_values)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = probs.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_probs_np = np.array(all_probs)
all_true_np = np.array(all_true)

# Top-3
top3_preds = np.argsort(all_probs_np, axis=1)[:, -3:]
top3_acc = np.mean([all_true_np[i] in top3_preds[i] for i in range(len(all_true_np))])

report = classification_report(all_true, all_preds, 
    target_names=[le.classes_[i] for i in sorted(set(all_true))],
    labels=sorted(set(all_true)),
    zero_division=0)

print('=' * 60)
print(f'FINAL — Accuracy: {np.mean(np.array(all_preds) == all_true_np):.1%}, Top-3: {top3_acc:.1%}')
print('=' * 60)
print(report)

# 결과 파일 저장 (세션 리셋 대비)
with open('/kaggle/working/hybrid_results.txt', 'w') as f:
    f.write(f'SoundTag AST Hybrid v2 Results\n')
    f.write(f'Dataset: {len(all_files)} tracks, {num_labels} genres\n')
    f.write(f'Train: {len(train_files)}, Val: {len(val_files)}\n\n')
    f.write(f'Accuracy: {np.mean(np.array(all_preds) == all_true_np):.1%}\n')
    f.write(f'Top-3: {top3_acc:.1%}\n\n')
    for log in results_log:
        f.write(log + '\n')
    f.write(f'\n{report}\n')

print('\n💾 hybrid_results.txt 저장 완료!')

FINAL — Accuracy: 46.0%, Top-3: 68.7%
                  precision    recall  f1-score   support

       Breakbeat       0.54      0.57      0.56       100
Contemporary R&B       0.50      0.14      0.22         7
           Disco       0.29      0.22      0.25        18
             DnB       0.40      0.39      0.40        56
         Dubstep       0.45      0.48      0.46        54
             EDM       0.71      0.55      0.62        53
      Electropop       0.36      0.23      0.28        35
            Funk       0.31      0.35      0.33        51
         Hip Hop       0.41      0.57      0.48        94
           House       0.47      0.52      0.50        63
            Jazz       0.41      0.33      0.36        40
       Latin Pop       0.43      0.48      0.45        42
           Lo fi       1.00      0.25      0.40         4
        Pop Rock       0.95      0.83      0.89        24
          Techno       0.47      0.36      0.41        45
          Trance       0.40      

In [10]:
# ═══ 셀 10: K-pop 곡 분석 ═══

# K-pop 데이터 경로 찾기
KPOP_DIR = None
for candidate in [
    '/kaggle/input/soundtag-kpop-previews',
    '/kaggle/input/datasets/zoebae/soundtag-kpop-previews',
]:
    if os.path.exists(candidate):
        KPOP_DIR = candidate
        break

if not KPOP_DIR:
    print('⚠️ soundtag-kpop-previews를 Add Input으로 추가하세요!')
else:
    # K-pop mp3 파일 수집
    kpop_files = sorted(Path(KPOP_DIR).rglob('*.mp3'))
    print(f'K-pop 파일: {len(kpop_files)}개')
    
    model.eval()
    kpop_results = []
    errors = 0
    
    for i, mp3 in enumerate(kpop_files):
        try:
            waveform, sr = torchaudio.load(str(mp3))
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.squeeze(0)
            if sr != 16000:
                waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
            
            max_len = 16000 * 10
            if len(waveform) > max_len:
                waveform = waveform[:max_len]
            elif len(waveform) < max_len:
                waveform = torch.nn.functional.pad(waveform, (0, max_len - len(waveform)))
            
            inputs = fe(waveform.numpy(), sampling_rate=16000, return_tensors='pt')
            input_values = inputs['input_values'].cuda()
            
            with torch.no_grad():
                probs = torch.softmax(model(input_values=input_values).logits, dim=-1)
                probs = probs.cpu().numpy()[0]
            
            top5 = [(str(le.classes_[j]), float(probs[j])) for j in np.argsort(probs)[-5:][::-1]]
            kpop_results.append({'filename': mp3.name, 'top5': top5})
            
            # 처음 30개만 출력
            if i < 30:
                print(f'  {mp3.name:45s} → {", ".join(f"{g}({s:.0%})" for g,s in top5[:3])}')
        
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f'  ❌ Error: {mp3.name} — {e}')
        
        if (i + 1) % 300 == 0:
            print(f'  [{i+1}/{len(kpop_files)}] 처리 중...')
    
    # K-pop 장르 분포
    print(f'\n{"=" * 60}')
    print(f'K-pop 장르 분포 ({len(kpop_results)}곡 분석)')
    print(f'{"=" * 60}')
    
    top1_dist = Counter(r['top5'][0][0] for r in kpop_results)
    for genre, count in top1_dist.most_common():
        pct = count / len(kpop_results) * 100
        print(f'  {genre:25s} {count:4d} ({pct:5.1f}%)')
    
    # K-pop 결과 저장
    with open('/kaggle/working/hybrid_kpop_analysis.json', 'w') as f:
        json.dump(kpop_results, f, ensure_ascii=False, indent=2)
    
    # 텍스트 결과 추가 저장
    with open('/kaggle/working/hybrid_results.txt', 'a') as f:
        f.write(f'\n\nK-pop 장르 분포 ({len(kpop_results)}곡):\n')
        for genre, count in top1_dist.most_common():
            pct = count / len(kpop_results) * 100
            f.write(f'  {genre:25s} {count:4d} ({pct:5.1f}%)\n')
    
    print(f'\n💾 hybrid_kpop_analysis.json 저장 완료!')
    print(f'   에러: {errors}개')

K-pop 파일: 1874개
  ATEEZ_1021374282.mp3                          → Hip Hop(51%), Dubstep(18%), Funk(8%)
  ATEEZ_1021374292.mp3                          → Dubstep(14%), House(12%), Trance(12%)
  ATEEZ_1021374302.mp3                          → Dubstep(23%), Hip Hop(23%), Electropop(14%)
  ATEEZ_1021374312.mp3                          → Hip Hop(31%), Dubstep(19%), Techno(9%)
  ATEEZ_1021374322.mp3                          → Dubstep(23%), House(14%), Hip Hop(13%)
  ATEEZ_1021374332.mp3                          → House(16%), Dubstep(14%), Electropop(12%)
  ATEEZ_1021374342.mp3                          → Latin Pop(28%), Hip Hop(24%), Lo fi(11%)
  ATEEZ_2041936627.mp3                          → Dubstep(29%), Techno(24%), EDM(15%)
  ATEEZ_2041936637.mp3                          → Funk(18%), House(15%), Latin Pop(11%)
  ATEEZ_2041936647.mp3                          → DnB(31%), Dubstep(31%), Breakbeat(14%)
  ATEEZ_2041936657.mp3                          → Pop Rock(58%), House(6%), Breakbeat(6%)
 

In [11]:
# ═══ 셀 11: 모델 저장 + 마무리 ═══

import zipfile

# 모델 + LabelEncoder 저장
torch.save(model.state_dict(), '/kaggle/working/ast_hybrid_final.pt')

import pickle
with open('/kaggle/working/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# zip으로 묶기
with zipfile.ZipFile('/kaggle/working/ast_hybrid_model.zip', 'w') as z:
    z.write('/kaggle/working/ast_hybrid_final.pt', 'ast_hybrid_final.pt')
    z.write('/kaggle/working/label_encoder.pkl', 'label_encoder.pkl')
    z.write('/kaggle/working/hybrid_results.txt', 'hybrid_results.txt')
    if os.path.exists('/kaggle/working/hybrid_kpop_analysis.json'):
        z.write('/kaggle/working/hybrid_kpop_analysis.json', 'hybrid_kpop_analysis.json')

print('=' * 60)
print('✅ ALL DONE!')
print('=' * 60)
print(f'\n저장된 파일:')
for f in sorted(Path('/kaggle/working').glob('*')):
    if f.is_file():
        size = f.stat().st_size / 1e6
        print(f'  {f.name:40s} {size:6.1f} MB')
print(f'\n📥 Output 탭에서 다운로드 가능!')

✅ ALL DONE!

저장된 파일:
  __notebook__.ipynb                          0.1 MB
  ast_hybrid_best.pt                        344.9 MB
  ast_hybrid_final.pt                       344.9 MB
  ast_hybrid_model.zip                      345.6 MB
  hybrid_kpop_analysis.json                   0.7 MB
  hybrid_results.txt                          0.0 MB
  label_encoder.pkl                           0.0 MB

📥 Output 탭에서 다운로드 가능!
